#  Pipeline de Vendas Diárias (ETL)
O objetivo deste notebook é transformar os dados crus de telefonia e pessoas em uma **Tabela de Faturamento Diário**,  particionada por quem é o Líder da Equipe.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum as _sum, round

# Ligando o motor
spark = SparkSession.builder.appName("Vendas_Diarias_ETL_Notebook").getOrCreate()

# Lendo as planilhas
df_telefonia = spark.read.csv("data/base_telefonia.csv", header=True, inferSchema=True)
df_pessoas = spark.read.csv("data/base_pessoas.csv", header=True, inferSchema=True)

print("Dados carregados na memória!")

Dados carregados na memória!


## Limpeza e Transformação
**Obs:** iremos trocar o valor "Vazio" pela palavra "Sem Lideranca".

In [5]:
# 1. Filtrar apenas as vendas verdadeiras
df_vendas = df_telefonia.filter((col("Valor venda").isNotNull()) & (col("Motivo") == "Venda"))

# 2. Pegar apenas o dia/mês/ano da data
df_vendas = df_vendas.withColumn("Data", to_date(col("inicio_ligacao")))

# 3. Cruzar com a tabela de pessoas para descobrir quem é o líder de quem
df_cruzado = df_vendas.join(
    df_pessoas.select("Username", col("Líder da Equipe").alias("lideranca")), 
    on="Username", 
    how="inner"
)

# 4. Onde o líder for vazio (nulo), preenchemos com "Sem Lideranca"
df_cruzado = df_cruzado.fillna("Sem Lideranca", subset=["lideranca"])

# 5. Calculando o faturamento somado por Dia e por Líder
df_vendas_diarias = df_cruzado.groupBy("Data", "lideranca") \
    .agg(round(_sum("Valor venda"), 2).alias("Valor_Total_Vendas"))

# Mostrando uma amostra de como a tabela final ficou
print("Amostra da Tabela Final:")
df_vendas_diarias.show(5)

Amostra da Tabela Final:
+----------+--------------------+------------------+
|      Data|           lideranca|Valor_Total_Vendas|
+----------+--------------------+------------------+
|2025-03-26| Cruz Teobaldo Souza|           5154.71|
|2025-03-18|Elizabete Galvão ...|           4309.25|
|2025-03-10|    Chico Zita Rocha|            5700.8|
|2025-03-05|Elizabete Galvão ...|           4742.67|
|2025-03-26|Dionísio Rosângel...|           5829.69|
+----------+--------------------+------------------+
only showing top 5 rows



## Guardando o Ficheiro
No formato  **Parquet**...

In [6]:
caminho_saida = "output/vendas_diarias_notebook_parquet"

# .repartition(1): Coloca todos os dados num único "cesto" antes de guardar, garantindo 1 ficheiro final.
# .partitionBy("lideranca"): Cria uma pasta para cada líder.
# .parquet(): Escolhe o formato...

df_vendas_diarias.repartition(1) \
    .write \
    .mode("overwrite") \
    .partitionBy("lideranca") \
    .parquet(caminho_saida)

print(f"Sucesso! Ficheiros guardados de forma otimizada na pasta '{caminho_saida}'.")

# Desliga o motor para libertar recursos do computador
spark.stop()

Sucesso! Ficheiros guardados de forma otimizada na pasta 'output/vendas_diarias_notebook_parquet'.
